# Fase 6 – Teste final (2025-26)

Decisões tomadas ANTES de correr este notebook (não mudam depois de ver o resultado):
- **Modelo final:** média geométrica de Ordinal GAP + Dixon-Coles + Boosting (escolhido na validação, notebook 02).
- **Referências:** Ordinal GAP sozinha, odds de abertura e de fecho, frequências.
- **Parâmetros:** GAP e Dixon-Coles afinados com 2018-19 a 2024-25; o boosting afina-se sozinho.
- **Regra:** o teste corre uma vez. O resultado fica registado como sair.

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

import sys
from pathlib import Path

# O notebook corre dentro da pasta "notebooks": sobe um nível e entra em "etl"
sys.path.insert(0, str(Path.cwd().parent / "etl"))

import configuracao as cfg
import func.avaliacao as av
import func.boosting as gb
import func.dixon_coles as dc
import func.ordinal as ordinal
from func.variaveis import colunas_do_modelo

with closing(sqlite3.connect(cfg.BASE_DADOS)) as ligacao:
    variaveis = pd.read_sql("SELECT * FROM variaveis ORDER BY Data, JogoID", ligacao)
    jogos = pd.read_sql("SELECT JogoID, Epoca, Data, EquipaCasa, EquipaFora, GolosCasa, GolosFora FROM jogos", ligacao)

USO = "Teste"
print("Época de teste:", list(variaveis.loc[variaveis["Uso"] == USO, "Epoca"].unique()))
print("Dixon-Coles: XI =", cfg.DC_XI, "RIDGE =", cfg.DC_RIDGE)

Época de teste: ['25-26']
Dixon-Coles: XI = 0.001 RIDGE = 2


In [2]:
MODELOS_FINAIS = ["Ordinal GAP", "Dixon-Coles", "Boosting"]
resultados = {}
resultados["Ordinal GAP"] = av.walk_forward(
    variaveis, ordinal.criar_previsor(["GapFalhadosDif", "GapBalizaDif", "GapCantosDif"]), USO)
resultados["Dixon-Coles"] = av.walk_forward(variaveis, dc.criar_previsor(jogos), USO)
previsor_gb = gb.criar_previsor(colunas_do_modelo(variaveis))
resultados["Boosting"] = av.walk_forward(variaveis, previsor_gb, USO)
print("Boosting, parâmetros escolhidos:", previsor_gb.escolhidos)

resultados["Modelo final"] = av.media_geometrica([resultados[m] for m in MODELOS_FINAIS])

Boosting, parâmetros escolhidos: {'25-26': {'learning_rate': 0.02, 'max_depth': 1, 'min_samples_leaf': 100, 'max_iter': 200}}


In [3]:
def frequencias(treino, teste):
    """Dizer sempre a percentagem de casa / empate / fora do treino."""
    f = treino["Resultado"].value_counts(normalize=True)[cfg.RESULTADOS_VALIDOS].to_numpy()
    return np.tile(f, (len(teste), 1))

def odds(tipo):
    colunas = [f"ProbOdds{tipo}Casa", f"ProbOdds{tipo}Empate", f"ProbOdds{tipo}Fora"]
    return lambda treino, teste: teste[colunas].to_numpy()

resultados["Frequências"] = av.walk_forward(variaveis, frequencias, USO)
resultados["Odds de abertura"] = av.walk_forward(variaveis, odds(""), USO)
resultados["Odds de fecho"] = av.walk_forward(variaveis, odds("Fecho"), USO)
av.resumo(resultados)

,Modelo,Acerto,LogLoss,Brier,RPS,ECE,LL 25-26
0,Odds de fecho,0.5752,0.9011,0.5324,0.1702,0.0611,0.9011
1,Odds de abertura,0.5654,0.9110,0.5389,0.1733,0.0704,0.9110
2,Modelo final,0.5556,0.9131,0.5398,0.1734,0.0529,0.9131
3,Ordinal GAP,0.5523,0.9192,0.5446,0.1755,0.0474,0.9192
4,Dixon-Coles,0.5556,0.9198,0.5428,0.1742,0.0576,0.9198
5,Boosting,0.5458,0.9201,0.5434,0.1754,0.0546,0.9201
6,Frequências,0.4118,1.0861,0.6580,0.2297,0.0209,1.0861


In [4]:
linhas = []
for outro in ["Ordinal GAP", "Odds de abertura", "Odds de fecho"]:
    media, minimo, maximo = av.comparar(resultados["Modelo final"], resultados[outro])
    linhas.append({"Modelo final contra": outro, "Diferença": media, "Mínimo": minimo, "Máximo": maximo,
                   "Pode ser sorte?": "Sim" if minimo < 0 < maximo else "Não"})
pd.DataFrame(linhas)

,Modelo final contra,Diferença,Mínimo,Máximo,Pode ser sorte?
0,Ordinal GAP,-0.0061,-0.0165,0.0047,Sim
1,Odds de abertura,0.0021,-0.0125,0.0173,Sim
2,Odds de fecho,0.0120,-0.0049,0.0294,Sim


In [5]:
pd.DataFrame({nome: av.ece_por_resultado(resultados[nome]) for nome in ["Modelo final", "Ordinal GAP", "Odds de fecho"]})

,Modelo final,Ordinal GAP,Odds de fecho
Resultado,,,
H,0.0651,0.0535,0.0582
D,0.0513,0.0507,0.0553
A,0.0422,0.0379,0.0697
Média,0.0529,0.0474,0.0611


In [6]:
pasta = cfg.PASTA_PROJETO / "dados"
av.resumo(resultados).to_csv(pasta / "resumo_teste.csv", index=False)
pd.concat([r.assign(Modelo=nome) for nome, r in resultados.items()]).to_csv(pasta / "previsoes_teste.csv", index=False)